# 01 · Define the task, then freeze the evaluation

**The decision this notebook supports:** what does "good" mean here, and how will I know if training helped?

This is the notebook people skip, and skipping it is why most fine-tuning projects cannot say whether they worked. The discipline is simple and almost never followed:

1. Write down what a good answer looks like **before** you have a model that produces one.
2. Build the evaluation set **before** you train, so you cannot tune against it by accident.
3. Split it into a development half you may look at and a held-out half you may not.

Everything downstream compares against the numbers you produce here.

In [ ]:
# Load this project's shared paths, configuration, scoring, and file helpers.
# sys controls Python's import search path; adding '..' assumes the notebook folder
# is the working directory. Import the project root, not src itself, so src/secrets.py
# does not replace Python's own secrets module. warnings controls diagnostic messages;
# the existing filter hides warnings, which can also hide useful compatibility clues.
# json handles JSON data; it is imported here for later cells (unused in this setup).
# CFG is a nested dictionary: CFG["model"]["name"] reads its model-name setting.
import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])

## The product task

Meridian Motors wants an assistant that answers customer questions about its three electric scooters using only the company's own documents.

The interesting part is not the answering. It is the **declining**. A product assistant that invents a price or a range figure is worse than one that says it does not know, because a wrong number reaches a customer as a promise. So the task has two halves:

- Answer accurately when the documents contain the answer.
- Refuse, clearly, when they do not.

That second half is why the evaluation set deliberately contains questions the documents cannot answer.

In [ ]:
# Inventory the source documents and inspect the structured facts used by scoring.
# json is Python's JSON reader/writer; dumps(..., indent=2) formats a Python
# object for human inspection. glob('*.md') selects Markdown files, while sorted
# makes the listing deterministic. No model or API is used in this cell.
# In f-strings, :32s aligns a string and :5d aligns an integer for readable columns.
import json
docs = sorted((lab.RAW).glob("*.md"))
print("source documents:")
for d in docs:
    print(f"  {d.name:32s} {len(d.read_text().split()):5d} words")

facts = lab.facts()
print(f"\nfacts extracted: {len(facts['facts'])}")
print("example fact:")
print(json.dumps(facts["facts"][0], indent=2))

## Look at the corpus before you model it

Read one document. You cannot define good answers for data you have not read, and this is the step that catches contradictions before they poison the evaluation.

In [ ]:
# Read the actual Arc 110 specification before defining expected answers.
# Path / filename constructs a file path; read_text() returns its contents.
# This is source evidence, not a model-generated summary.
print((lab.RAW / "spec_meridian_arc_110.md").read_text())

In [ ]:
# Read the behavioral requirements as well as the numeric specification.
# The brand guide defines how to answer, when to decline, and what not to claim.
# It will later inform prompts and training examples.
print((lab.RAW / "brand_guide.md").read_text())

The brand guide matters more than it looks. *"Never round a range figure upward"* and *"say so directly"* are behavioural requirements, not style notes. They are exactly the kind of thing supervised fine-tuning teaches and preference tuning enforces.

## The evaluation set

Already built by `src/build_data.py` from the same facts, so questions and answers cannot drift apart. Sixty questions: forty-five answerable from the documents, fifteen not.

The split is 60% development and 40% held-out. **You may inspect development failures freely. Look at held-out results once, at the end.** If you inspect held-out failures and then change something, that set has become development data and its numbers no longer mean what you think.

In [ ]:
# Load the existing evaluation splits and count their composition; this cell
# actually reads both files. Its displayed examples come only from dev.
# Reserve held-out question content and failures for the final evaluation;
# repeatedly tuning against them would invalidate that independent check.
# A boolean adds as 1 for True and 0 for False, so sum(answerable) counts positives.
dev = lab.eval_set("dev")
held = lab.eval_set("heldout")

for name, s in [("dev", dev), ("heldout", held)]:
    a = sum(q["answerable"] for q in s)
    print(f"{name:8s} n={len(s):3d}  answerable={a:3d}  unanswerable={len(s)-a:3d}")

# A list comprehension [q for q in dev if ...] filters questions; [:3] takes
# three examples. q['unit'] or '' substitutes an empty string when the unit
# is absent, preventing the display from printing None.
print("\n--- three answerable ---")
for q in [q for q in dev if q["answerable"]][:3]:
    print(f"  {q['question']}\n     expect: {q['expected_value']} {q['unit'] or ''}")
print("\n--- three unanswerable ---")
for q in [q for q in dev if not q["answerable"]][:3]:
    print(f"  {q['question']}")

## Scoring, worked by hand before it is automated

Four outcomes matter, and conflating them is the most common evaluation mistake:

| | answerable question | unanswerable question |
|---|---|---|
| **gave the right value** | correct | not possible |
| **gave a wrong value** | **hallucination** | **hallucination** |
| **declined** | over-refusal | **correct** |

Note the asymmetry. Declining is a failure on the left and the desired behaviour on the right. A single "accuracy" number hides that completely, which is why `lab.summarise` reports them separately.

Work through four examples by hand before trusting the function.

In [ ]:
# Exercise the shared scorer on deliberately chosen replies before trusting
# aggregate scores. The two list comprehensions pick one answerable and one
# unanswerable dev question. Refusal has opposite meanings in those two groups.
# reply[:48] shortens the displayed text only; scoring receives the full reply.
# This scorer checks values and refusal keywords, not full semantic correctness:
# a right number attached to the wrong model or an extra false claim can slip by.
fidx = lab.fact_index()
q_ans = [q for q in dev if q["answerable"]][0]
q_un  = [q for q in dev if not q["answerable"]][0]

print("ANSWERABLE:", q_ans["question"], "| truth:", q_ans["expected_value"])
for reply in [f"The certified range is {q_ans['expected_value']} {q_ans['unit'] or ''}.",
              "It is about 200 km.",
              "I don't have that information."]:
    print(f"   {reply[:48]:50s} -> {lab.score_answer(reply, q_ans, fidx)}")

print("\nUNANSWERABLE:", q_un["question"])
for reply in ["I don't have information on that in the documents.",
              "The resale value is around 60% after three years."]:
    print(f"   {reply[:48]:50s} -> {lab.score_answer(reply, q_un, fidx)}")

Read those outputs carefully. In the unanswerable case the confident reply scores `hallucinated: True` even though it sounds entirely reasonable. **That is the behaviour a fluent model is rewarded for during pretraining and which we now have to train out.**

## Declare the bar before you see any result

Write it down now, while you have no stake in the outcome. Edit the numbers if you disagree, but do it before running notebook 02.

In [ ]:
# Declare quality thresholds before observing model results to reduce goalpost
# moving. BAR is a dictionary of metric names to proportions, not percentages.
# lab.save_run writes this decision to runs/00_bar.json; rerunning overwrites that
# named artifact. .items() iterates over (key, value) pairs for the printed record.
BAR = {
    "accuracy_answerable":          0.80,  # gets the right number 8 times in 10
    "hallucination_rate_answerable":0.10,  # invents a number at most 1 time in 10
    "appropriate_refusal_rate":     0.80,  # declines 8 of 10 unanswerable questions
    "max_over_refusal":             0.15,  # but does not become uselessly cautious
}
lab.save_run("00_bar", {"bar": BAR,
    "note": "Declared in notebook 01 before any model was measured."})
for k, v in BAR.items():
    print(f"{k:32s} {v}")

## What you should be able to say now

- Why the evaluation set is built before the model, and what contamination would do to every later number.
- Why refusal is a success on one half of the set and a failure on the other.
- The specific bar you set, and that you set it blind.

**Next:** notebook 02 measures the stock model, then finds the ceiling of prompting. If prompting clears the bar, the correct engineering decision is to stop and not fine-tune at all.